In [ ]:
# Use Case: Real-time Language Translation
# Business Impact: Global communication, customer support, content localization

from transformers import pipeline
import gradio as gr

# Initialize translation pipelines (free)
# Multiple language pairs supported
translation_models = {
    "English to French": "Helsinki-NLP/opus-mt-en-fr",
    "English to German": "Helsinki-NLP/opus-mt-en-de", 
    "English to Spanish": "Helsinki-NLP/opus-mt-en-es",
    "French to English": "Helsinki-NLP/opus-mt-fr-en",
    "German to English": "Helsinki-NLP/opus-mt-de-en",
    "Spanish to English": "Helsinki-NLP/opus-mt-es-en"
}

# Cache translators to avoid reloading
translators = {}

def get_translator(language_pair):
    if language_pair not in translators:
        translators[language_pair] = pipeline("translation", 
                                            model=translation_models[language_pair], 
                                            framework="pt")
    return translators[language_pair]

def translate_text(text, source_lang, target_lang):
    if not text.strip():
        return "Please enter text to translate!"
    
    # Create language pair key
    language_pair = f"{source_lang} to {target_lang}"
    
    if language_pair not in translation_models:
        return f"❌ Translation pair '{language_pair}' not supported yet."
    
    try:
        # Get translator and translate
        translator = get_translator(language_pair)
        result = translator(text)
        
        # Calculate some stats
        original_words = len(text.split())
        translated_words = len(result[0]['translation_text'].split())
        
        # Language flags for visual appeal
        flags = {
            "English": "🇺🇸", "French": "🇫🇷", "German": "🇩🇪", 
            "Spanish": "🇪🇸", "Italian": "🇮🇹", "Portuguese": "🇵🇹"
        }
        
        source_flag = flags.get(source_lang, "🌐")
        target_flag = flags.get(target_lang, "🌐")
        
        return f"""
## {source_flag} → {target_flag} Translation Complete

### Original ({source_lang}):
{text}

### Translation ({target_lang}):
**{result[0]['translation_text']}**

---
📊 **Stats:** {original_words} words → {translated_words} words
🤖 **Model:** {translation_models[language_pair].split('/')[-1]}
        """
        
    except Exception as e:
        return f"❌ Error: {str(e)}"

# Business use case examples
business_examples = [
    ["Welcome to our company! We provide excellent customer service.", "English", "Spanish"],
    ["Bonjour, comment puis-je vous aider aujourd'hui?", "French", "English"], 
    ["Unser Produkt ist von höchster Qualität.", "German", "English"],
    ["Thank you for your purchase. Your order will be shipped tomorrow.", "English", "French"]
]

interface = gr.Interface(
    fn=translate_text,
    inputs=[
        gr.Textbox(
            label="Text to Translate",
            placeholder="Enter text in any supported language...",
            lines=4
        ),
        gr.Dropdown(
            choices=["English", "French", "German", "Spanish"],
            label="Source Language",
            value="English"
        ),
        gr.Dropdown(
            choices=["English", "French", "German", "Spanish"],
            label="Target Language", 
            value="Spanish"
        )
    ],
    outputs=gr.Markdown(),
    title="🌍 AI Translation Assistant",
    description="Translate text between multiple languages instantly using AI",
    examples=business_examples
)

if __name__ == "__main__":
    interface.launch(share=True)